# EmporiUm Sales Territory Analysis

This project analyzes EmporiUm sales data using Python to compare sales performance between two Northeast territories. The analysis focuses on the territories managed by Shruti Reddy and Erbayne Middleton.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [3]:
store_sales = pd.read_csv("StoreSales.csv")
store_sales.head()
store_sales.info()


<class 'pandas.DataFrame'>
RangeIndex: 335129 entries, 0 to 335128
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Transaction Date  335129 non-null  str    
 1   Store ID          335129 non-null  int64  
 2   RewardsID         34943 non-null   float64
 3   Prod Num          335129 non-null  str    
 4   Sale Amount       335129 non-null  float64
dtypes: float64(2), int64(1), str(2)
memory usage: 12.8 MB


In [4]:
store_detail = pd.read_csv("StoreDetail.csv")
store_detail.head()
store_detail.info()


<class 'pandas.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Store Location     111 non-null    str  
 1   State              111 non-null    str  
 2   Store ID           111 non-null    int64
 3   Territory Manager  111 non-null    str  
 4   Region             111 non-null    str  
 5   Region Director    111 non-null    str  
dtypes: int64(1), str(5)
memory usage: 5.3 KB


In [5]:
products = pd.read_csv("Products (1).csv")
products.head()
products.info()


<class 'pandas.DataFrame'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Prod Num       669 non-null    str  
 1   Product        669 non-null    str  
 2   CategoryID     669 non-null    int64
 3   SubcategoryID  669 non-null    str  
dtypes: int64(1), str(3)
memory usage: 21.0 KB


In [6]:
product_categories = pd.read_csv("ProductCategories.csv")
product_categories.head()
product_categories.info()


<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   CategoryID     52 non-null     int64
 1   Category       52 non-null     str  
 2   SubcategoryID  52 non-null     str  
 3   Subcategory    52 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1.8 KB


In [9]:
customer_list = pd.read_csv("customer_list.csv", sep="|")
customer_list.columns = customer_list.columns.str.strip()

customer_list.head()
customer_list.info()


<class 'pandas.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   cust_id      521 non-null    int64
 1   date         521 non-null    str  
 2   time         521 non-null    str  
 3   name         521 non-null    str  
 4   email        521 non-null    str  
 5   phone        520 non-null    str  
 6   sms-opt-out  520 non-null    str  
dtypes: int64(1), str(6)
memory usage: 28.6 KB


## Core Marketing Analysis


### 1. Territory managers, store IDs, and store cities

This section identifies the two territory managers being analyzed and lists the stores included in each territory.


In [12]:
territory_managers = ["Shruti Reddy", "Erbayne Middleton"]
territory_stores = store_detail[store_detail["Territory Manager"].isin(territory_managers)].copy()
territory_stores = territory_stores.sort_values(["Territory Manager", "Store ID"])
territory_stores[["Territory Manager", "Store ID", "Store Location", "State", "Region"]]

,Territory Manager,Store ID,Store Location,State,Region
37,Erbayne Middleton,818,Bangor,Maine,Northeast
38,Erbayne Middleton,819,Bar Harbor,Maine,Northeast
39,Erbayne Middleton,820,Kennebunkport,Maine,Northeast
40,Erbayne Middleton,821,Lewiston,Maine,Northeast
41,Erbayne Middleton,822,Orono,Maine,Northeast
42,Erbayne Middleton,823,South Portland,Maine,Northeast
43,Shruti Reddy,731,Annapolis,Maryland,Northeast
44,Shruti Reddy,732,Back River,Maryland,Northeast
45,Shruti Reddy,733,Baltimore,Maryland,Northeast
46,Shruti Reddy,734,Germantown,Maryland,Northeast


Shruti Reddy and Erbayne Middleton are the two territory managers included in this analysis. Shruti Reddy manages stores in Maryland, while Erbayne Middleton manages stores in Maine, and both territories are in the Northeast region.


### 2. Monthly total revenue by territory

This section compares monthly total in-store sales revenue for the two assigned territories across the full time period in the dataset.


In [14]:
territory_sales = store_sales.merge(store_detail, on="Store ID", how="left")
territory_sales = territory_sales[territory_sales["Territory Manager"].isin(territory_managers)].copy()
territory_sales["Transaction Date"] = pd.to_datetime(territory_sales["Transaction Date"])
territory_sales["month"] = territory_sales["Transaction Date"].dt.to_period("M")
territory_sales["Sale Amount"] = pd.to_numeric(territory_sales["Sale Amount"], errors="coerce")
monthly_revenue = (
    territory_sales
    .groupby(["month", "Territory Manager"])["Sale Amount"]
    .sum()
    .reset_index()
    .sort_values(["month", "Territory Manager"])
)

monthly_revenue


,month,Territory Manager,Sale Amount
0,2022-01,Erbayne Middleton,15700.31
1,2022-01,Shruti Reddy,190064.90
2,2022-02,Erbayne Middleton,21008.29
3,2022-02,Shruti Reddy,197529.18
4,2022-03,Erbayne Middleton,23173.23
...,...,...,...
91,2025-10,Shruti Reddy,359699.69
92,2025-11,Erbayne Middleton,57222.33
93,2025-11,Shruti Reddy,304194.50
94,2025-12,Erbayne Middleton,65695.32


This table shows the monthly sales revenue for each territory across the full period in the dataset. It can be used to compare overall sales patterns and identify which territory generated stronger monthly performance over time.


### 3. Store sales performance ranking
This section ranks stores within each assigned territory based on total sales revenue and identifies the top-performing stores.


In [16]:
store_performance = (
    territory_sales
    .groupby(["Territory Manager", "Store ID", "Store Location", "State"])["Sale Amount"]
    .sum()
    .reset_index()
    .sort_values(["Territory Manager", "Sale Amount"], ascending=[True, False])
)    
store_performance["Store Rank"] = (
    store_performance
    .groupby("Territory Manager")["Sale Amount"]
    .rank(method="dense", ascending=False)
)    

store_performance

,Territory Manager,Store ID,Store Location,State,Sale Amount,Store Rank
5,Erbayne Middleton,823,South Portland,Maine,332611.76,1.0
4,Erbayne Middleton,822,Orono,Maine,330505.47,2.0
2,Erbayne Middleton,820,Kennebunkport,Maine,321998.55,3.0
3,Erbayne Middleton,821,Lewiston,Maine,303761.91,4.0
0,Erbayne Middleton,818,Bangor,Maine,300919.98,5.0
1,Erbayne Middleton,819,Bar Harbor,Maine,287452.08,6.0
11,Shruti Reddy,736,North Harford,Maryland,8708119.00,1.0
9,Shruti Reddy,734,Germantown,Maryland,584675.92,2.0
12,Shruti Reddy,737,Parkville,Maryland,320441.24,3.0
10,Shruti Reddy,735,Howard,Maryland,319394.58,4.0


In [17]:
top_stores = store_performance[store_performance["Store Rank"] <= 5]
top_stores


,Territory Manager,Store ID,Store Location,State,Sale Amount,Store Rank
5,Erbayne Middleton,823,South Portland,Maine,332611.76,1.0
4,Erbayne Middleton,822,Orono,Maine,330505.47,2.0
2,Erbayne Middleton,820,Kennebunkport,Maine,321998.55,3.0
3,Erbayne Middleton,821,Lewiston,Maine,303761.91,4.0
0,Erbayne Middleton,818,Bangor,Maine,300919.98,5.0
11,Shruti Reddy,736,North Harford,Maryland,8708119.00,1.0
9,Shruti Reddy,734,Germantown,Maryland,584675.92,2.0
12,Shruti Reddy,737,Parkville,Maryland,320441.24,3.0
10,Shruti Reddy,735,Howard,Maryland,319394.58,4.0
14,Shruti Reddy,739,Ridgely,Maryland,318511.04,5.0


This ranking shows the strongest-performing stores in each territory based on total revenue. The stores with the highest total sales can be considered the top-performing stores in their territory.


### 4. Top customers in each sales territory
This section compares the customer list with rewards member purchases in the sales data to identify the highest-spending customers in each territory.


In [23]:
customer_list.columns = customer_list.columns.str.strip()

rewards_sales = territory_sales[territory_sales["RewardsID"].notna()].copy()

top_customers = rewards_sales.merge(
    customer_list,
    left_on="RewardsID",
    right_on="cust_id",
    how="left"
)

top_customers = (
    top_customers
    .groupby(["Territory Manager", "RewardsID", "name", "email"])["Sale Amount"]
    .sum()
    .reset_index()
    .sort_values(["Territory Manager", "Sale Amount"], ascending=[True, False])
)

top_customers["Customer Rank"] = (
    top_customers
    .groupby("Territory Manager")["Sale Amount"]
    .rank(method="dense", ascending=False)
)

top_customers[top_customers["Customer Rank"] <= 10]


,Territory Manager,RewardsID,name,email,Sale Amount,Customer Rank
384,Erbayne Middleton,421.0,Rosita,rosita@sesamestreet.org,3637.75,1.0
80,Erbayne Middleton,89.0,Karen Walker,karen@willandgrace.nyc,3362.83,2.0
313,Erbayne Middleton,345.0,Maddy Perez,maddy@easthighland.high,2706.42,3.0
8,Erbayne Middleton,9.0,Mike H.,mike@centralperk.coffee,2695.22,4.0
186,Erbayne Middleton,208.0,Philip Banks,unclephil@banksresidence.la,2662.84,5.0
437,Erbayne Middleton,478.0,AJ Soprano,aj@northjersey.net,2574.08,6.0
88,Erbayne Middleton,99.0,Trent Lane,trent@lawndale.high,2558.21,7.0
320,Erbayne Middleton,353.0,Laura Palmer,laura@twinpeaks.wa,2488.19,8.0
325,Erbayne Middleton,358.0,James Hurley,james@twinpeaks.wa,2482.78,9.0
164,Erbayne Middleton,185.0,Nipsey,nipsey@wzhup.det,2112.13,10.0


This table identifies the top rewards customers in each territory based on total spending. These customers may represent strong candidates for retention, targeted promotions, or loyalty-focused marketing efforts.


### 5. Monthly transactions and revenue by product category
This section examines how many transactions occurred each month by product category in each territory, and how much revenue each category generated.


In [24]:
territory_product_sales = (
    territory_sales
    .merge(products, on="Prod Num", how="left")
    .merge(product_categories, on=["CategoryID", "SubcategoryID"], how="left")
)
territory_product_sales["month"] = territory_product_sales["Transaction Date"].dt.to_period("M")
territory_product_sales.head()

,Transaction Date,Store ID,RewardsID,Prod Num,Sale Amount,Store Location,State,Territory Manager,Region,Region Director,month,Product,CategoryID,SubcategoryID,Category,Subcategory
0,2022-01-01,731,NaN,105384-A,23.44,Annapolis,Maryland,Shruti Reddy,Northeast,Michael Jarvis,2022-01,Silver Brush Limited Ruby Satin Brushes (Set o...,115,115-pai,Art Supplies,Paint Brushes
1,2022-01-01,731,NaN,105384-A,23.44,Annapolis,Maryland,Shruti Reddy,Northeast,Michael Jarvis,2022-01,Silver Brush Limited Ruby Satin Brushes (Set o...,115,115-pai,Art Supplies,Paints
2,2022-01-01,733,20.0,105385-M,27.52,Baltimore,Maryland,Shruti Reddy,Northeast,Michael Jarvis,2022-01,LTCA Year Up Foam Finger (Black),130,130-spo,Apparel and Merchandise,Sports and Outdoor Gear
3,2022-01-01,734,NaN,105386-M,25.00,Germantown,Maryland,Shruti Reddy,Northeast,Michael Jarvis,2022-01,LTCA Year Up Graduation Sash,130,130-mem,Apparel and Merchandise,Memorabilia and Collectibles
4,2022-01-01,735,NaN,105349-M,8.00,Howard,Maryland,Shruti Reddy,Northeast,Michael Jarvis,2022-01,LTCA Year Up Gel Pen (Pack of 2),130,130-off,Apparel and Merchandise,Office and Study Supplies


In [25]:
monthly_category_transactions = (
    territory_product_sales
    .groupby(["month", "Territory Manager", "Category"])
    .size()
    .reset_index(name="Transaction Count")
    .sort_values(["month", "Territory Manager", "Transaction Count"], ascending=[True, True, False])
)

monthly_category_transactions


,month,Territory Manager,Category,Transaction Count
4,2022-01,Erbayne Middleton,Technology & Accessories,35
1,2022-01,Erbayne Middleton,Art Supplies,33
0,2022-01,Erbayne Middleton,Apparel and Merchandise,30
3,2022-01,Erbayne Middleton,Stationery and Supplies,29
5,2022-01,Erbayne Middleton,Textbooks,24
...,...,...,...,...
573,2025-12,Shruti Reddy,Stationery and Supplies,467
570,2025-12,Shruti Reddy,Apparel and Merchandise,410
571,2025-12,Shruti Reddy,Art Supplies,408
572,2025-12,Shruti Reddy,Books (General),387


In [26]:
monthly_category_revenue = (
    territory_product_sales
    .groupby(["month", "Territory Manager", "Category"])["Sale Amount"]
    .sum()
    .reset_index()
    .sort_values(["month", "Territory Manager", "Sale Amount"], ascending=[True, True, False])
)

monthly_category_revenue



,month,Territory Manager,Category,Sale Amount
4,2022-01,Erbayne Middleton,Technology & Accessories,9007.64
5,2022-01,Erbayne Middleton,Textbooks,4273.52
0,2022-01,Erbayne Middleton,Apparel and Merchandise,958.97
1,2022-01,Erbayne Middleton,Art Supplies,874.56
2,2022-01,Erbayne Middleton,Books (General),370.52
...,...,...,...,...
575,2025-12,Shruti Reddy,Textbooks,42818.37
570,2025-12,Shruti Reddy,Apparel and Merchandise,12759.16
571,2025-12,Shruti Reddy,Art Supplies,12541.45
572,2025-12,Shruti Reddy,Books (General),12034.48


In [27]:
category_summary = (
    territory_product_sales
    .groupby(["Territory Manager", "Category"])
    .agg(
        total_transactions=("Sale Amount", "size"),
        total_revenue=("Sale Amount", "sum")
    )
    .reset_index()
    .sort_values(["Territory Manager", "total_revenue"], ascending=[True, False])
)

category_summary


,Territory Manager,Category,total_transactions,total_revenue
4,Erbayne Middleton,Technology & Accessories,2848,1359289.84
5,Erbayne Middleton,Textbooks,1800,309330.88
0,Erbayne Middleton,Apparel and Merchandise,2552,80611.53
1,Erbayne Middleton,Art Supplies,2585,80083.23
2,Erbayne Middleton,Books (General),1050,29730.25
3,Erbayne Middleton,Stationery and Supplies,2763,27717.10
10,Shruti Reddy,Technology & Accessories,16836,7826937.84
11,Shruti Reddy,Textbooks,11639,2021177.87
6,Shruti Reddy,Apparel and Merchandise,15547,500587.47
8,Shruti Reddy,Books (General),15335,494845.76


The category summaries show which product categories drive the most transactions and revenue in each territory. Categories with strong transaction counts and strong revenue suggest the most popular products, while categories with lower performance may represent opportunities for growth through targeted marketing or promotion.


### 6. Recommendation for next quarter

Based on the analysis, marketing attention in the next quarter should focus on the categories and stores that show the strongest sales potential while also addressing weaker areas with growth opportunity.

If one territory consistently generates stronger monthly revenue and category performance, its successful product mix and promotional approach may offer strategies that can be applied to the other territory. Special attention should be given to top-performing categories, top stores, and high-value rewards customers, since these areas are likely to produce the greatest return on marketing effort.
